In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
prompt = "Narendra Modi is the prime"

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# Logits for the NEXT token
next_token_logits = outputs.logits[:, -1, :]

# Convert logits to probabilities
probs = torch.softmax(next_token_logits, dim=-1)

# Top 5 predictions
top_probs, top_indices = torch.topk(probs, k=5)

print(f"Prompt: {prompt}\n")

print("Top 5 Predicted Tokens:\n")

for token_id, prob in zip(top_indices[0], top_probs[0]):
    token = tokenizer.decode(token_id)
    print(f"{token!r:<15} Probability: {prob.item():.4f}")

Prompt: Narendra Modi is the prime

Top 5 Predicted Tokens:

' minister'     Probability: 0.7649
' ministerial'  Probability: 0.1473
' m'            Probability: 0.0144
' suspect'      Probability: 0.0095
' example'      Probability: 0.0081


Softmax implementation

In [ ]:
import numpy as np
logits = np.array([-2.5, 1.2, 0.3])

softmax = []
denominator = 0
for i in range(len(logits)):
  denominator = np.exp(logits[i]) + denominator
for i in range(len(logits)):
  softmax.append(np.exp(logits[i])/denominator)

print(softmax)

[np.float64(0.017273558421520868), np.float64(0.6986688748566597), np.float64(0.2840575667218194)]


Temperatur, Top-k, Top-p

In [ ]:
!pip install -q transformers

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

torch_device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# add the EOS token as PAD token to avoid warnings
model = AutoModelForCausalLM.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id).to(torch_device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
#Greedy Search
model_inputs = tokenizer('I always wanted to read a book', return_tensors='pt').to(torch_device)

greedy_output = model.generate(**model_inputs, max_new_tokens=40)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I always wanted to read a book about the history of the world, but I didn't have the time to do it. I was too busy with my family and my work to do it. I was too busy with my family and


In [ ]:
#Beam Search
model_inputs = tokenizer('I always wanted to read a book', return_tensors='pt').to(torch_device)

greedy_output = model.generate(**model_inputs, max_new_tokens=40,num_beams=5,
    early_stopping=True)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

print("\n\nWith no repeat ngrams")
greedy_output = model.generate(**model_inputs, max_new_tokens=40,num_beams=5,no_repeat_ngram_size=2,
    early_stopping=True)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I always wanted to read a book, but I didn't know what to do with it. I didn't know what to do with it. I didn't know what to do with it. I didn't know what to do with


 with no repeat ngrams
Output:
----------------------------------------------------------------------------------------------------
I always wanted to read a book, but I didn't know what to do with it. So I went to a bookstore and bought a copy of the book. I read it, and I was like, "Oh my God,


In [ ]:
#Temperature
from transformers import set_seed
set_seed(42)

# activate sampling and deactivate top_k by setting top_k sampling to 0
sample_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature = 0.7
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I always wanted to read a book about the importance of science. I remember the first time I read it was a couple of years ago. I was reading it at a book fair in Washington D.C. and I was thinking,


In [ ]:
#top-k sampling
from transformers import set_seed
set_seed(42)

# activate sampling and deactivate top_k by setting top_k sampling to 0
sample_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    do_sample=True,
    top_k=50,
    temperature = 1.0
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I always wanted to read a book with me," a teacher told me. "When I saw the book, I felt like we would spend an entire week in the library." One of the teachers told me several times that she had gotten


In [ ]:
#top - p sampling
from transformers import set_seed
set_seed(42)

# activate sampling and deactivate top_k by setting top_k sampling to 0
sample_output = model.generate(
    **model_inputs,
    max_new_tokens=40,
    do_sample=True,
    top_p = 0.96
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I always wanted to read a book with me," a teacher told me. "When I saw the book, I felt like we would have an incredible connection. I always wanted to read an American novel with me. I liked the American


Streamlit APP

In [1]:
!pip install -q streamlit
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 102.9 MB/s eta 0:00:00


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
def load_model(model_name="gpt2-medium"):
  model_name = "gpt2-medium"

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForCausalLM.from_pretrained(model_name)
  return tokenizer, model


In [22]:
def tokenize(tokenizer,prompt):
  torch_device = "cuda" if torch.cuda.is_available() else "cpu"
  model_inputs = tokenizer(prompt, return_tensors='pt').to(torch_device)
  token_count = len(model_inputs['input_ids'][0])
  return model_inputs,token_count

In [23]:
def temp_model_output(temperature,model_inputs):
  with torch.no_grad():
    sample_output = model.generate(
      **model_inputs,
      max_new_tokens=40,
      do_sample=True,
      temperature = temperature
    )
  return sample_output


In [26]:
prompt = "I am very good at reading"
tokenizer,model = load_model()
model_inputs,token_count = tokenize(tokenizer,prompt)
print("Token count:",token_count)
sample_output = temp_model_output(0.2,model_inputs)
print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Token count: 5
Output:
----------------------------------------------------------------------------------------------------
I love sadhana. I love sadhana. I love sadhana. I love sadhana. I love sadhana. I love sadhana. I love sadhana. I love sad


In [32]:
%%writefile app.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import streamlit as st
def load_model(model_name="gpt2-medium"):
  model_name = "gpt2-medium"

  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForCausalLM.from_pretrained(model_name)
  return tokenizer, model

def tokenize(tokenizer,prompt):
  torch_device = "cuda" if torch.cuda.is_available() else "cpu"
  model_inputs = tokenizer(prompt, return_tensors='pt').to(torch_device)
  token_count = len(model_inputs['input_ids'][0])
  return model_inputs,token_count
def temp_model_output(temperature,model_inputs):
  with torch.no_grad():
    sample_output = model.generate(
      **model_inputs,
      max_new_tokens=40,
      do_sample=True,
      temperature = temperature
    )
  return sample_output
st.title("LLM Inference Playground")
prompt = st.text_input("Enter your Prompt:")
tokenizer,model = load_model()
model_inputs,token_count = tokenize(tokenizer,prompt)

temperature = st.sidebar.slider(
    "Temperature",
    min_value=0.0,
    max_value=2.0,
    value=0.5,
    step=0.1
)
if st.button("Generate"):
  sample_output = temp_model_output(temperature,model_inputs)
  st.write(tokenizer.decode(sample_output[0], skip_special_tokens=True))
  st.write("Token count:",token_count)






Overwriting app.py


In [36]:
%%writefile app.py
import time
import torch
import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM

st.set_page_config(page_title="LLM Inference Playground", layout="wide")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

@st.cache_resource
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    model.eval()
    return tokenizer, model

st.title("🧠 LLM Inference Playground")

st.sidebar.header("Inference Settings")
model_name = st.sidebar.selectbox("Model", ["gpt2","gpt2-medium","gpt2-large"], index=1)
temperature = st.sidebar.slider("Temperature",0.0,2.0,0.7,0.1)
top_k = st.sidebar.slider("Top-K",1,100,50)
top_p = st.sidebar.slider("Top-P",0.1,1.0,0.95,0.05)
max_tokens = st.sidebar.slider("Max New Tokens",10,200,50,10)
use_cache = st.sidebar.checkbox("Use KV Cache", True)

tokenizer, model = load_model(model_name)
prompt = st.text_area("Prompt", value="The future of Artificial Intelligence is")

def generate(prompt, temp):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    start = time.time()
    output = model.generate(
        **inputs,
        do_sample=True,
        temperature=temp,
        top_k=top_k,
        top_p=top_p,
        max_new_tokens=max_tokens,
        use_cache=use_cache,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - start
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    input_tokens = len(inputs["input_ids"][0])
    output_tokens = len(output[0]) - input_tokens
    return text, input_tokens, output_tokens, elapsed

if st.button("Generate"):
    text, in_tok, out_tok, elapsed = generate(prompt, temperature)
    st.subheader("Generated Output")
    st.write(text)

    c1,c2,c3,c4 = st.columns(4)
    c1.metric("Input Tokens", in_tok)
    c2.metric("Output Tokens", out_tok)
    c3.metric("Total Tokens", in_tok + out_tok)
    c4.metric("Inference Time", f"{elapsed:.2f}s")


Overwriting app.py


In [37]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501

⠙⠹⠸your url is: https://cool-books-sin.loca.lt
^C
